In [10]:
import os
from pathlib import Path

import pandas as pd
from src import analysis, util
from src.analysis import evaluation

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [11]:
# ======================================================================
# Configuration
# ======================================================================
# Old Models trained on Dataset with scalar kappa
# Models trained on Dataset with scalar kappa
# modes(32, 32), hidden_channels=64
model = "FNO_lhs_var10_plog100_seed1_20251208_230304"
model = "FNO_lhs_var20_plog100_seed1001_20251210_163827"
model = "FNO_lhs_var40_plog100_seed2001_20251211_154034"
model = "FNO_lhs_var80_plog100_seed3001_20251212_125538"

# modes(12, 12), hidden_channels=24
model = "FNO_lhs_var80_plog100_seed3001_20260103_121739"

parts = model.split("_")
dataset_name_id = "_".join(parts[1:5])

dataset_names = [
    dataset_name_id,
    # "lhs_var20_plog100_seed1001",
    # "lhs_var40_plog100_seed2001",
    # "lhs_var80_plog100_seed3001",
    "lhs_var160_plog100_seed4001",
]

In [12]:
# ======================================================================
# Models trained on Dataset with tensor kappa, porosity field
# and varying pressure boundary condition
# =====================================================================
# --- FNO ---
# FNO small
model = "FNO_m12x12_h24_l4_lhs_var80_seed3001_20260109_201346"
# FNO big

# --- PI-FNO ---
# PI-FNO small
model = "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704"
# PI-FNO big

# --- U-NO ---
# U-NO small

# U-NO big

# --- PI-U-NO ---
# PI-U-NO small

# PI-U-NO big

dataset_names = [
    "lhs_var80_seed3001",
    "lhs_var120_seed4001",
    # "lhs_var160_seed5001",
]

In [13]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
checkpoint_path = Path(f"../data/processed/{model}/best_model_state_dict.pt")

# build cases paths automatically
dataset_cases_paths = {name: Path(f"../../data/raw/{name}/cases") for name in dataset_names}

# build save roots automatically
save_roots = {dataset_names[0]: Path(f"../data/processed/{model}/analysis/id")}

save_roots.update({name: Path(f"../data/processed/{model}/analysis/ood") / name for name in dataset_names[1:]})

In [ ]:
def run_or_load_artifacts_evaluation(
    dataset_name: str,
    save_root: Path,
    dataset_path: Path,
) -> tuple[pd.DataFrame, Path]:
    """Load an existing Parquet artifact for the given dataset, or create it if missing."""
    parquet_path = save_root / f"{dataset_name}.parquet"

    if parquet_path.exists():
        print(f"[INFO] Found existing parquet -> loading: {parquet_path}")
        df = pd.read_parquet(parquet_path)
        print(f"[INFO] Loaded df with {len(df)} rows")
        return df, parquet_path

    print(f"[INFO] Creating artifacts for: {dataset_name}")

    model, loader, processor, device = analysis.analysis_interference.load_inference_context(
        dataset_path=dataset_path,
        checkpoint_path=checkpoint_path,
        batch_size=1,
    )

    df, parquet_path = analysis.analysis_artifacts.generate_artifacts(
        model=model,
        loader=loader,
        processor=processor,
        device=device,
        save_root=save_root,
        dataset_name=dataset_name,
    )

    print("[INFO] Artifact generation done.")
    return df, parquet_path

In [15]:
# ---------------------------------------------------------------------
# Generate artifacts
# ---------------------------------------------------------------------
datasets_raw = {}
parquet_paths = {}

for name in dataset_names:
    df_raw, parquet_path = run_or_load_artifacts_evaluation(
        dataset_name=name,
        save_root=save_roots[name],
        dataset_path=dataset_cases_paths[name],
    )
    datasets_raw[name] = df_raw
    parquet_paths[name] = parquet_path

[INFO] Creating artifacts for: lhs_var80_seed3001


/home/mambauser/workspace/model_training/src/analysis/analysis_interference.py:251: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint

[INFO] Detected grid: NY=251, NX=401
[INFO] Found existing parquet -> loading: ../data/processed/PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704/analysis/id/lhs_var80_seed3001.parquet
[INFO] Loaded df with 1000 rows
[INFO] Creating artifacts for: lhs_var120_seed4001
[INFO] Detected grid: NY=251, NX=401
[INFO] Found existing parquet -> loading: ../data/processed/PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704/analysis/ood/lhs_var120_seed4001/lhs_var120_seed4001.parquet
[INFO] Loaded df with 1000 rows


/home/mambauser/workspace/model_training/src/analysis/analysis_interference.py:251: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint

In [16]:
datasets_eval = {name: evaluation.evaluation_dataframe.load_and_build_eval_df(parquet_paths[name]) for name in dataset_names}

In [17]:
datasets_eval["lhs_var80_seed3001"]

,case_index,npz_path,l2,rel_l2,kappa_names,export_delimiter,export_file_base,fields_present_porosity,fields_present_pressure_bc,fields_present_tensor,...,geometry_Lx,geometry_Ly,geometry_dx,geometry_dy,geometry_nx,geometry_ny,geometry_res,paths_csv,paths_json,timestamp
0,1,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,745.987610,0.005277,"[kxx, kyy, kxy]",;,case_0001,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 08:54:47
1,2,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,904.693604,0.009852,"[kxx, kyy, kxy]",;,case_0002,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 08:55:07
2,3,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,1716.355103,0.014405,"[kxx, kyy, kxy]",;,case_0003,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 08:55:26
3,4,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,917.232422,0.006761,"[kxx, kyy, kxy]",;,case_0004,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 08:55:47
4,5,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,797.067505,0.010215,"[kxx, kyy, kxy]",;,case_0005,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 08:56:07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,1795.409424,0.019804,"[kxx, kyy, kxy]",;,case_0996,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 14:30:18
996,997,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,805.237488,0.013604,"[kxx, kyy, kxy]",;,case_0997,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 14:30:38
997,998,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,901.371643,0.011390,"[kxx, kyy, kxy]",;,case_0998,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 14:30:58
998,999,../data/processed/PI-FNO_m12x12_h24_l4_lamPhys...,2991.488525,0.019999,"[kxx, kyy, kxy]",;,case_0999,True,True,True,...,1.2,0.75,0.003,0.003,401,251,0.003,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,C:\Users\rino.albertin\OneDrive - OST\2 MSE FH...,2026-01-06 14:31:18


In [18]:
# =====================================================================
# TOGGLE SETUP
# =====================================================================
toggle = util.util_nb.make_toggle_shortcut(dfs=datasets_eval)

# =====================================================================
# 1. GLOBAL ERROR ANALYSIS (HOW LARGE IS THE ERROR OVERALL?)
# =====================================================================
global_error_analysis_plots = [
    toggle("1-1. Global error metrics", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_metrics),
    toggle("1-2. Global error distribution", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_error_distribution),
    toggle("1-3. GT vs Prediction (mean)", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_gt_vs_pred),
    toggle("1-4. Mean error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_mean_error_maps),
    toggle("1-5. Std error maps", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_std_error_maps),
    toggle("1-6. Error frequency spectrum", evaluation.evaluation_plot.evaluation_plot_global_error_analysis.plot_global_error_frequency_spectrum),
]

# =====================================================================
# 2. ERROR DECOMPOSITION (WHERE DOES THE ERROR OCCUR?)
# =====================================================================
error_decomposition_plots = [
    toggle("2-1. Error vs |GT| magnitude", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_gt_magnitude),
    toggle("2-2. Boundary vs interior error", evaluation.evaluation_plot.evaluation_plot_error_decomposition.plot_error_vs_boundary_distance),
]

# =====================================================================
# 3. PHYSICAL CONSISTENCY (IS THE SOLUTION PHYSICALLY VALID?)
# =====================================================================
physical_consistency_plots = [
    toggle("3-1. Velocity divergence (∇·u)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_velocity_divergence),
    toggle("3-2. Mass conservation error map", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_mass_conservation_error_map),
    toggle("3-3. Pressure boundary consistency (p_bc)", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_pressure_bc_consistency),
    toggle("3-4. Darcy-Brinkman operator residual", evaluation.evaluation_plot.evaluation_plot_physical_consistency.plot_brinkman_residual),
]

# =====================================================================
# 4. ERROR SENSITIVITY (WHICH INPUT PARAMETERS DRIVE THE ERROR?)
# =====================================================================
error_sensitivity_plots = [
    toggle(
        "4-1. Parameter-error correlation (heatmap)", evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_parameter_error_heatmap
    ),
    toggle(
        "4-2. Error vs input parameter (binned trend)", evaluation.evaluation_plot.evaluation_plot_parameter_sensitivity.plot_error_vs_parameter_trend
    ),
]

# =====================================================================
# 5. SAMPLE VIEWER (WHAT DOES THE MODEL PREDICT IN INDIVIDUAL CASES?)
# =====================================================================
sample_viewer_plots = [
    toggle("5-1. Sample GT vs Prediction", evaluation.evaluation_plot.evaluation_plot_sample_viewer.plot_sample_prediction_overview),
    toggle("5-2. Kappa tensor with error overlay", evaluation.evaluation_plot.evaluation_plot_sample_viewer.plot_sample_kappa_tensor_with_overlay),
]

# =====================================================================
# 6. OUTLIER AND EXTREME CASE ANALYSIS (WHEN AND WHY DOES THE MODEL FAIL?)
# =====================================================================
outlier_plots = [
    toggle("6-1. Worst per-channel cases (tables)", evaluation.evaluation_plot.evaluation_plot_outlier_analysis.plot_outlier_tables_per_channel),
    toggle("6-2. Worst per-channel cases (field plots)", evaluation.evaluation_plot.evaluation_plot_outlier_analysis.plot_outlier_cases_per_channel),
    toggle("6-3. Extreme input parameters (table view)", evaluation.evaluation_plot.evaluation_plot_outlier_analysis.plot_extreme_input_table),
    toggle("6-4. Extreme input parameter cases (field plots)", evaluation.evaluation_plot.evaluation_plot_outlier_analysis.plot_extreme_input_cases),
]

# =====================================================================
# SECTIONS UND TABS
# =====================================================================
sections = [
    util.util_nb.make_dropdown_section(global_error_analysis_plots),
    util.util_nb.make_dropdown_section(error_decomposition_plots),
    util.util_nb.make_dropdown_section(physical_consistency_plots),
    util.util_nb.make_dropdown_section(error_sensitivity_plots),
    util.util_nb.make_dropdown_section(sample_viewer_plots),
    util.util_nb.make_dropdown_section(outlier_plots),
]

tab_titles = [
    "1. Global Error Analysis",
    "2. Error Decomposition",
    "3. Physical Consistency",
    "4. Error Sensitivity (Input Parameters)",
    "5. Sample Viewer",
    "6. Outlier and Extreme Case Analysis",
]

# =====================================================================
# FINAL PANEL
# =====================================================================
evaluation_panel = util.util_nb.make_lazy_panel_with_tabs(
    sections,
    tab_titles=tab_titles,
    open_btn_text=f"{model} - Open Evaluation",
    close_btn_text="Close",
)

display(evaluation_panel)

Output()